In [1]:
from usecase import Usecase
uc = Usecase()

In [2]:
uc.specification.get_config()

'4-ft|15-10-1000-10-40-1000-6|10-100-1-1-1|True-True|True-50-5|False-0.74-0.25|False-10|True-5-20|False|100-200-300|1-3|64-256|0.5-1.0-2.0-5.0-10.0'

python -m notebook

In [2]:
from stream import Stream, TrafficHandler

stream = Stream.generate(uc.configuration)
stream.to_dict()

{'name': 'stream-11',
 'type': 'isochronous',
 'interval': 100000,
 'maxFramesPerInterval': 1,
 'maxFrameSize': 136,
 'maxLatency': 500000,
 'tlkrPssblPlcmnt': [],
 'lstnsrPssblPlcmnt': [],
 'earliest_transmit_offset': 0,
 'latest_transmit_offset': 50000,
 'jitter': 0,
 'lifespanCycles': 8310000}

In [3]:
stream = TrafficHandler.embed_to_topology(stream, uc.configuration, uc.graph, network_load=[])
stream.to_dict()

{'name': 'stream-11',
 'type': 'isochronous',
 'interval': 100000,
 'maxFramesPerInterval': 1,
 'maxFrameSize': 136,
 'maxLatency': 500000,
 'tlkrPssblPlcmnt': ['host-11-p2'],
 'lstnsrPssblPlcmnt': [['host-16-p3']],
 'earliest_transmit_offset': 0,
 'latest_transmit_offset': 50000,
 'jitter': 0,
 'pathLists': [['host-11-p2.Port1',
   'switch-p2-e2.Port1',
   'switch-p2-a1.Port1',
   'switch-c1.Port4',
   'switch-p3-a1.Port4',
   'switch-p3-e2.Port4',
   'host-16-p3']],
 'lifespanCycles': 8310000}

In [ ]:
def build_pruned_graph(self, graph: nx.DiGraph):
    """
    Simple pruning rule:

    IF pathLists exists:
        keep only ports in pathLists (and their edges)
        → everything else is implicitly removed

    ELSE:
        remove concurrent ports of:
            - tlkrPssblPlcmnt (talkers)
            - lstnsrPssblPlcmnt (listeners)
    """

    from topology_handler import TopologyHandler

    # --------------------------------------------------
    # CASE 1: path-based pruning (STRICT MODE)
    # --------------------------------------------------
    if self.pathLists:

        active_ports = set()

        # keep only ports in pathLists (exclude last element = listener node)
        for path in self.pathLists:
            for hop in path[:-1]:
                active_ports.add(hop)

        # remove everything not in active_ports
        pruned = nx.DiGraph()
        pruned.add_nodes_from(graph.nodes(data=True))
        for u, v, data in graph.edges(data=True):
            src_port = data.get("sourcePort")
            if src_port in active_ports:
                pruned.add_edge(u, v, **data)

        isolated_nodes = [n for n, d in pruned.degree() if d < 1]
        if isolated_nodes:
            pruned.remove_nodes_from(isolated_nodes)

        return pruned

    # --------------------------------------------------
    # CASE 2: free routing mode (concurrency pruning)
    # --------------------------------------------------
    edges_to_remove = []

    # talker concurrency removal
    edges_to_remove += TopologyHandler.get_concurrent_ports(
        graph,
        self.tlkrPssblPlcmnt,
        typ="talker"
    )

    # listener concurrency removal
    edges_to_remove += TopologyHandler.get_concurrent_ports(
        graph,
        [l for sub in self.lstnsrPssblPlcmnt for l in sub],
        typ="listener"
    )

    # rebuild graph
    pruned = nx.DiGraph()
    pruned.add_nodes_from(graph.nodes(data=True))

    for u, v, data in graph.edges(data=True):
        if (u, v, data) in edges_to_remove:
            continue
        pruned.add_edge(u, v, **data)

    isolated_nodes = [n for n, d in pruned.degree() if d < 1]
    if isolated_nodes:
        pruned.remove_nodes_from(isolated_nodes)

    return pruned